# Generate Customer Feature Matrix

This notebook runs the full preprocessing + feature engineering pipeline to produce a
**per-customer statistics matrix** that is used to train the cross-sell VAE.

Pipeline stages:
1. **Load** raw transaction CSV (`read_lumen_csv` with encoding fallback chain)
2. **Inspect** raw data schema
3. **Preprocess** rows and customers (`Preprocessor.fit_transform`)
4. **Build features** — aggregate per-customer statistics (`build_customer_feature_matrix`)
5. **Inspect** the resulting feature matrix
6. **Save** the finalized matrix to CSV for training

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ is on sys.path so package imports resolve without pip install -e
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_processing.preprocessing import Preprocessor, PreprocessorConfig, NullRemovalRule
from embeddings.customer_features import (
    CustomerFeatureConfig,
    build_customer_feature_matrix,
    describe_feature_groups,
    read_lumen_csv,
    schema_report,
)

print("Imports OK")

## Configuration

Adjust the paths and preprocessing knobs below before running the rest of the notebook.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
RAW_CSV_PATH   = Path("C:/Users/dijan/Desktop/LUMEN_DS.csv")
OUTPUT_CSV_PATH = Path("C:/Users/dijan/Desktop/customer_features.csv")

# ── Preprocessing knobs ────────────────────────────────────────────────────
# Minimum transactions per customer to keep them in the dataset
MIN_PURCHASES_PER_CUSTOMER = 3
# Minimum distinct items a customer must have bought
MIN_ITEMS_PER_CUSTOMER = 2
# Minimum rows for an item to be retained (helps remove one-off catalogue noise)
MIN_ITEM_ROWS = 5

# ── Feature engineering knobs ──────────────────────────────────────────────
# Number of product groups per bucket (affects bucket feature count)
BUCKET_SIZE = 5
# Top-N most common values kept for channel share columns
TOP_N_CHANNEL_VALUES = 12
# Drop a feature column if more than this fraction of customers are missing it
FEATURE_MAX_MISSING_RATIO = 0.85

# Set to a small integer (e.g. 50_000) for a quick smoke-test; None = full data
NROWS_DEBUG: int | None = None

print(f"Raw CSV   : {RAW_CSV_PATH}")
print(f"Output CSV: {OUTPUT_CSV_PATH}")

---
## Step 1 — Load raw data

In [ ]:
raw_df = read_lumen_csv(RAW_CSV_PATH, nrows=NROWS_DEBUG)

print(f"Shape  : {raw_df.shape}  ({raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns)")
print(f"Memory : {raw_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
raw_df.head(3)

In [ ]:
# Column-level null / type overview (sorted by null ratio descending)
schema_report(raw_df)

In [ ]:
# Quick sanity: how many unique customers and items are there?
n_customers = raw_df["CustomerID"].nunique()
n_items     = raw_df["Item Code"].nunique()
print(f"Unique customers : {n_customers:,}")
print(f"Unique items     : {n_items:,}")
print(f"Rows per customer (median): {raw_df.groupby('CustomerID').size().median():.0f}")

---
## Step 2 — Preprocess

`Preprocessor.fit_transform` runs all enabled stages in order:

| Stage | What it does |
|---|---|
| `missing_identifier_rows` | Drop rows with null `CustomerID` or `Item Code` |
| `date_parsing` | Parse date columns to `datetime64` |
| `item_mapping` | Assign a stable integer index to every item code |
| `customer_null_rules` | Remove customers that exceed column-level null thresholds |
| `price_filter` | Drop rows where invoiced price > TX price (data quality) |
| `item_frequency_filters` | Remove rare items (too few rows / too few unique customers) |
| `customer_frequency_filters` | Remove customers with too few purchases or unique items |
| `final_item_reindex` | Re-assigns item indices after frequency filtering |
| `impute` | Per-customer median imputation of remaining missing values |

In [ ]:
preprocessor_config = PreprocessorConfig(
    customer_col="CustomerID",
    item_col="Item Code",
    date_columns=("Order Date", "Invoice Date"),
    price_col="Invoiced price",
    tx_price_col="Invoiced price (TX)",
    null_rules=(
        NullRemovalRule(
            columns=("CustomerID",),
            max_null_fraction=0.0,
            name="require_customer_id",
        ),
        NullRemovalRule(
            columns=("Invoiced price", "Invoiced price (TX)"),
            max_null_fraction=0.5,
            name="price_null_threshold",
        ),
    ),
    min_num_purchases_per_customer=MIN_PURCHASES_PER_CUSTOMER,
    min_num_items_per_customer=MIN_ITEMS_PER_CUSTOMER,
    min_num_purchases_per_item_rows=MIN_ITEM_ROWS,
    drop_price_gt_tx=True,
    impute=True,
)

preprocessor = Preprocessor(config=preprocessor_config)
clean_df, idx2item = preprocessor.fit_transform(raw_df)

print(f"Clean shape : {clean_df.shape}  ({clean_df.shape[0]:,} rows)")
print(f"Customers   : {clean_df['CustomerID'].nunique():,}")
print(f"Items       : {clean_df['Item Code'].nunique():,}")

In [ ]:
# Detailed report for each preprocessing stage
report = preprocessor._report
print(f"Initial  : {report['initial_rows']:>8,} rows | {report['initial_customers']:>6,} customers")
print(f"Final    : {report['final_rows']:>8,} rows | {report['final_customers']:>6,} customers")
print()
for stage_name, stage_data in report.get("stages", {}).items():
    rows_removed     = stage_data.get("rows_removed", 0)
    customers_removed = stage_data.get("customers_removed", 0)
    if rows_removed or customers_removed:
        print(f"  [{stage_name}]  -rows: {rows_removed:,}  -customers: {customers_removed:,}")

---
## Step 3 — Build customer feature matrix

For each customer `build_customer_feature_matrix` computes:

| Feature group | Prefix | Description |
|---|---|---|
| Aggregates | `agg__` | Sum / mean / median / std / max of numeric transaction columns |
| Activity | `activity__` | Row count, unique items/groups, entropy measures |
| Time | `time__` | Recency days, tenure days |
| Full shares | `share__` | Fraction of rows per category value (product family, region, …) |
| Bucket shares | `bucketshare__` | Revenue share across product-group frequency buckets |
| Bucket magnitudes | `bucket__` | Raw revenue / count per bucket |
| Family hierarchy | `family__` / `familygroup__` | Product-family revenue & count; share of product group within family |

In [ ]:
feature_config = CustomerFeatureConfig(
    customer_col="CustomerID",
    item_col="Item Code",
    product_group_col="Product group",
    product_family_col="Product family",
    revenue_col="Invoiced price",
    bucket_size=BUCKET_SIZE,
    top_n_channel_values=TOP_N_CHANNEL_VALUES,
    min_channel_count=100,
    feature_max_missing_ratio=FEATURE_MAX_MISSING_RATIO,
    feature_imputation_strategy="median",
)

feature_matrix, metadata = build_customer_feature_matrix(
    clean_df,
    feature_config,
    return_metadata=True,
)

print(f"Feature matrix shape : {feature_matrix.shape}")
print(f"  Customers          : {feature_matrix.shape[0]:,}")
print(f"  Features           : {feature_matrix.shape[1]:,}")
print(f"  Columns dropped    : {len(metadata['dropped_feature_columns'])}")

In [ ]:
# Feature counts per group
groups = metadata["feature_groups"]
group_counts = {k: len(v) for k, v in groups.items()}
print("Feature counts by group:")
for group, count in sorted(group_counts.items(), key=lambda x: -x[1]):
    print(f"  {group:<22} : {count}")

In [ ]:
# Null / missing check on the finalized matrix (should be 0 after imputation)
null_counts = feature_matrix.isna().sum()
has_nulls = null_counts[null_counts > 0]
if has_nulls.empty:
    print("No missing values in the finalized feature matrix.")
else:
    print(f"{len(has_nulls)} columns still have missing values:")
    display(has_nulls)

In [ ]:
# Quick statistical summary of key aggregate features
agg_cols = [c for c in feature_matrix.columns if c.startswith("agg__")][:12]
feature_matrix[agg_cols].describe().round(3)

In [ ]:
# Distribution of the most informative activity features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

feature_matrix["activity__row_count"].hist(bins=50, ax=axes[0], color="steelblue")
axes[0].set_title("Transactions per customer")
axes[0].set_xlabel("row count")

feature_matrix["activity__unique_items"].hist(bins=50, ax=axes[1], color="darkorange")
axes[1].set_title("Unique items per customer")
axes[1].set_xlabel("unique items")

feature_matrix["time__invoice_recency_days"].hist(bins=50, ax=axes[2], color="forestgreen")
axes[2].set_title("Invoice recency (days)")
axes[2].set_xlabel("days since last invoice")

plt.tight_layout()
plt.show()

---
## Step 4 — Save feature matrix

The matrix is saved as a CSV with the `CustomerID` as the index column.
This file is the input for training the cross-sell VAE.

In [ ]:
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
feature_matrix.to_csv(OUTPUT_CSV_PATH, index=True)

size_mb = OUTPUT_CSV_PATH.stat().st_size / 1e6
print(f"Saved  : {OUTPUT_CSV_PATH}")
print(f"Size   : {size_mb:.2f} MB")
print(f"Shape  : {feature_matrix.shape[0]:,} customers × {feature_matrix.shape[1]:,} features")

In [ ]:
# Optional: also save metadata (bucket map, dropped columns, imputation stats)
META_PATH = OUTPUT_CSV_PATH.with_suffix(".meta.json")

serializable_meta = {
    "feature_groups": metadata["feature_groups"],
    "dropped_feature_columns": metadata["dropped_feature_columns"],
    "raw_feature_count": len(metadata["raw_feature_columns"]),
    "final_feature_count": len(metadata["final_feature_columns"]),
    "imputation_strategy": metadata["imputation_strategy"],
    "feature_max_missing_ratio": metadata["feature_max_missing_ratio"],
    # bucket_map keys are ints; convert for JSON
    "bucket_map": {str(k): v for k, v in metadata["bucket_map"].items()},
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(serializable_meta, f, indent=2, default=str)

print(f"Metadata : {META_PATH}")

---
## Next steps

With the customer feature matrix saved you can now:

1. Open `cross_sell_vae_upgrade_walkthrough.ipynb` and point it to `customer_features.csv`.
2. Train the cross-sell `RecommendationAwareVAE` on the customer vectors.
3. Use the latent embeddings for downstream clustering / recommendation.